# Module 20 — Inference Optimization Under 8–24 GB RAM

Companion notebook to [`docs/modules/20_inference_optimization_under_8_24gb_ram.md`](../docs/modules/20_inference_optimization_under_8_24gb_ram.md).

Mostly a playbook, not a rebuild: quantization choice, context budgeting, streaming, caching,
KV cache behavior, concurrency control, and queueing are all reused unchanged from Modules 4,
6, 6.5, and 12. This notebook demonstrates the genuinely new pieces — a model router, a
fallback chain, a benchmark harness, prompt compression, and a performance dashboard — plus
the composed context-budgeting demo that ties two existing budgeters together.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_20"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. Lab 1: benchmark harness — real simulated-latency measurement

In [2]:
from benchmark_harness_demo import run_lab as run_benchmark_lab, results_to_markdown as benchmark_to_markdown

benchmark_results = await run_benchmark_lab()
print(benchmark_to_markdown(benchmark_results))

# Lab 1 - benchmark harness

| Config | Samples | Mean latency (ms) | p95 latency (ms) | Mean tokens/sec |
|---|---:|---:|---:|---:|
| q4_small_model | 5 | 5.68 | 5.74 | 176.2 |
| q8_medium_model | 5 | 21.07 | 21.09 | 94.9 |
| fp16_large_model | 5 | 61.18 | 61.23 | 98.1 |



## 2. Lab 2: context budgeter — composing Module 12 + Module 8.5 unchanged

In [3]:
from context_budget_demo import run_lab as run_context_budget_lab, result_to_markdown as context_budget_to_markdown

context_budget_result = run_context_budget_lab()
print(context_budget_to_markdown(context_budget_result))

# Lab 2 - context budgeter (composing Module 12 + Module 8.5 unchanged)

- Context window: 4096
- Conversation history budget: 3204 (used 27)
- Remaining for retrieval: 3177
- Packed 2 chunk(s): ['password-reset-guide', 'account-security-faq']



## 3. Lab 3: model router

In [4]:
from model_router_demo import run_lab as run_router_lab, results_to_markdown as router_to_markdown

router_results = await run_router_lab()
print(router_to_markdown(router_results))

# Lab 3 - model router

- **simple classification** -> small (no escalation signal present - the small model tier is cheaper and sufficient) -> (small model) done
- **long document summary** -> large (prompt token count (3500) exceeds the small-model threshold (2000)) -> (large model) done
- **multi-step agent plan** -> large (task requires multi-step reasoning - the small model tier is not reliable for this) -> (large model) done
- **tool-calling request** -> large (task requires tool calls - the large model tier has more reliable tool-call formatting) -> (large model) done



## 4. Lab 4: fallback model

In [5]:
from fallback_demo import run_lab as run_fallback_lab, result_to_markdown as fallback_to_markdown

fallback_result = await run_fallback_lab()
print(fallback_to_markdown(fallback_result))

# Lab 4 - fallback model

- Fell back to runtime index 1 after 2 attempt(s): "billing (from fallback runtime)"
- Non-retryable validation error propagated without fallback: True (secondary runtime call count: 0)
- Every runtime down raised NoRuntimesAvailable: True



## 5. Labs 5-6: queueing and streaming — Module 6.5 + Module 6 unchanged

In [6]:
from queueing_streaming_demo import run_lab as run_queue_lab, result_to_markdown as queue_to_markdown

queue_result = await run_queue_lab()
print(queue_to_markdown(queue_result))

# Labs 5-6 - queueing and streaming (Module 6.5 + Module 6 unchanged)

- Ran 4 concurrent streaming requests through a max_concurrent=2 bounded queue
- Streamed chunk counts per request: [3, 3, 3, 3]
- First request's reassembled text: "billing category ticket"
- Queue drained cleanly: running=0, waiting=0



## 6. Prompt compression — real, deterministic, non-LLM shrinking

In [7]:
from local_ai_core.optimization.prompt_compression import compress_prompt

noisy_prompt = (
    "Please classify the following support ticket.\n"
    "Please classify the following support ticket.\n"
    "Ticket: I was charged twice for my renewal.\n"
    "\n\n\n"
    "Respond with one category word."
)
compression_result = compress_prompt(noisy_prompt)
print(f"Original tokens (heuristic): {compression_result.original_token_estimate}")
print(f"Compressed tokens (heuristic): {compression_result.compressed_token_estimate}")
print(f"Reduction ratio: {compression_result.reduction_ratio:.2%}")
print("---")
print(compression_result.compressed_text)

Original tokens (heuristic): 32
Compressed tokens (heuristic): 25
Reduction ratio: 21.88%
---
Please classify the following support ticket.
Ticket: I was charged twice for my renewal.

Respond with one category word.


## 7. Lab 7: performance dashboard

In [8]:
from performance_dashboard_demo import run_lab as run_dashboard_lab, result_to_markdown as dashboard_to_markdown

dashboard_result = await run_dashboard_lab()
print(dashboard_to_markdown(dashboard_result))

# Lab 7 - performance dashboard

- Requests: 10 (errors: 2, error rate: 20%)
- Mean latency: 7.13ms
- p50 latency: 8.89ms
- p95 latency: 9.16ms
- Mean tokens/sec: 224.4



## Summary

- Benchmark harness: real latency/tokens-per-second measurement over simulated runtimes.
- Context budgeter: two existing budgeters (Module 12's RAG context, Module 8.5's conversation
  history) composed against one shared context window — no new package code needed.
- Model router: a real, testable escalation decision, any single strong signal routes to the
  large model tier.
- Fallback model: a real ordered runtime chain, retryable errors fall through, non-retryable
  errors propagate immediately.
- Queueing and streaming: Module 6.5's bounded queue and Module 6's streaming, reused unchanged.
- Prompt compression: a real, lossless-ish, non-LLM reduction, distinct from Module 8.5's
  LLM-based summarization.
- Performance dashboard: real p50/p95 latency and error-rate aggregation over live traffic.